# Project: Analyzing Car Reviews with LLMs

**Course: LLMs with PyTorch — Capstone Project**

## Brief

Car-Vision Inc, an auto dealership chain, wants to use large language models (LLMs) to
build an automated system that can process **customer reviews** at scale, without any
task-specific training of their own. This project builds a small proof-of-concept using
pretrained models from the Hugging Face Hub, wired together with the `transformers`
`pipeline()` API — the same pattern used throughout this course's other notebooks
(`introtollm.ipynb`), applied to four separate NLP tasks on the same small corpus of
reviews:

1. **Sentiment classification** — is each review positive or negative?
2. **Translation** — translate one review into Spanish and check quality with BLEU
3. **Extractive question answering** — pull a specific answer out of a review
4. **Summarization** — condense a long review into one or two sentences

> **Compute note.** Every cell below downloads a pretrained model from the Hugging Face
> Hub the first time it runs, and several of them are large enough to want a GPU. If you
> are running this offline or on CPU-only hardware, read the cells for the pattern —
> `pipeline(task, model=...)` in, structured output out — rather than expecting every
> cell to execute quickly.

In [ ]:
import pandas as pd
from transformers import pipeline, logging as hf_logging

hf_logging.set_verbosity_error()   # keep the download/progress noise down

## 1. The data

A small, hand-labelled sample of car reviews, in the same `review / label` shape the
real DataCamp project ships (`car_reviews.csv`). Two are clearly positive, two clearly
negative — enough to sanity-check a classifier without needing a GPU cluster.

In [ ]:
data = pd.DataFrame({
    "review": [
        "I am very satisfied with my Nissan Kicks, it's a great car for the value. "
        "I would recommend this car to anyone looking for a car in the sub 25k range.",
        "The car is fine but I am disappointed with the amount of legroom in the rear "
        "seats. Also the engine is quite loud when accelerating and doesn't feel "
        "particularly powerful.",
        "My first foreign car. Love it, I would buy another one again. It is very "
        "quiet and comfortable to drive. It's the right size for me, not too big and "
        "not too small.",
        "I've come across numerous reviews stating that the fuel economy of this "
        "vehicle is not as impressive as expected, and I must say I agree with these "
        "assessments. The mileage falls short of what was anticipated.",
    ],
    "label": ["POSITIVE", "NEGATIVE", "POSITIVE", "NEGATIVE"],
})
data

## 2. Task 1 — Sentiment classification

`pipeline("sentiment-analysis")` loads a model already fine-tuned for exactly this task
(SST-2, a movie-review sentiment dataset) and applies it directly — **zero training of
our own required**. We compare its predictions against our hand-assigned labels.

In [ ]:
classifier = pipeline("sentiment-analysis")

predictions = classifier(data["review"].tolist())
predicted_labels = [p["label"] for p in predictions]

data["predicted_label"] = predicted_labels
data["confidence"] = [round(p["score"], 4) for p in predictions]
data[["review", "label", "predicted_label", "confidence"]]

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(data["label"], data["predicted_label"])
f1 = f1_score(data["label"], data["predicted_label"], pos_label="POSITIVE")
print(f"accuracy: {accuracy:.3f}")
print(f"F1 (positive class): {f1:.3f}")
print()
print("Off-the-shelf sentiment models trained on movie reviews transfer reasonably well")
print("to car reviews -- 'satisfied', 'disappointed', 'love it' carry sentiment that is")
print("largely domain-independent. Where they struggle is nuanced, mixed-sentiment text")
print("(the second review praises nothing but is really a MILD complaint, not outrage --")
print("a fine-tuned domain model would calibrate that better than a generic one).")

## 3. Task 2 — Translation

Translate the first review into Spanish with a pretrained MarianMT model, and score the
translation against a short reference translation using **BLEU** — the standard n-gram
overlap metric for machine translation quality.

In [ ]:
translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es")

first_review = data["review"].iloc[0]
translated = translator(first_review, max_length=100)[0]["translation_text"]

print("EN:", first_review)
print("ES:", translated)

In [ ]:
from nltk.translate.bleu_score import sentence_bleu

# A short human reference translation of the first two sentences, for scoring
reference = (
    "Estoy muy satisfecho con mi Nissan Kicks, es un gran coche por el precio. "
    "Lo recomendaria a cualquiera que busque un coche por menos de 25 mil."
)

reference_tokens = [reference.lower().split()]
candidate_tokens = translated.lower().split()

bleu = sentence_bleu(reference_tokens, candidate_tokens)
print(f"BLEU score: {bleu:.4f}")
print()
print("BLEU rewards exact n-gram overlap with the reference, so paraphrased-but-correct")
print("translations still score lower than they 'deserve' -- it is a useful automatic")
print("proxy, not a substitute for a bilingual reviewer signing off on real deployments.")

## 4. Task 3 — Extractive question answering

Given a review and a question, an extractive-QA model finds the **span of the original
text** that answers the question — it does not generate new text, only points at it.

In [ ]:
qa_pipeline = pipeline("question-answering")

context = data["review"].iloc[1]
question = "What did the reviewer dislike about the car?"

answer = qa_pipeline(question=question, context=context)

print("Context :", context)
print("Question:", question)
print(f"Answer  : '{answer['answer']}'  (confidence {answer['score']:.4f})")

## 5. Task 4 — Summarization

A long review gets condensed to a couple of sentences with a pretrained summarization
model (BART, fine-tuned on CNN/DailyMail news summarization — again, no domain-specific
training needed for a first pass).

In [ ]:
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

long_review = data["review"].iloc[3]
summary = summarizer(long_review, max_length=40, min_length=10, do_sample=False)[0]["summary_text"]

print("Original  :", long_review)
print("\nSummary   :", summary)

## 6. Putting it together

In production, Car-Vision Inc would wrap these four pipelines behind a single service:
new reviews come in, get classified and (optionally) summarized automatically, get
routed to a human agent with the sentiment and a QA-extracted "key complaint" attached,
and get translated on demand for regional teams. None of the four models needed a
single labelled car-review example to work reasonably well — the value of the pretrained
LLM approach is exactly that zero-shot transfer.

In [ ]:
def analyze_review(text):
    '''Run all four pipelines on one review and return a structured summary.'''
    sentiment = classifier(text)[0]
    return {
        "sentiment": sentiment["label"],
        "confidence": round(sentiment["score"], 3),
        "summary": summarizer(text, max_length=40, min_length=10, do_sample=False)[0]["summary_text"]
                   if len(text.split()) > 25 else text,
    }

for review in data["review"]:
    result = analyze_review(review)
    print(result)
    print()

## What to try next

* Fine-tune the sentiment classifier on a larger, labelled set of *car-specific* reviews
  (see `finetuning_llama2.ipynb` in this same folder for the fine-tuning workflow, applied
  there to instruction-following rather than classification).
* Replace the single-sentence BLEU reference with a proper held-out translation test set.
* Add a "key complaint" extraction step: run the QA pipeline with a fixed battery of
  questions ("What did the reviewer dislike?", "What would they change?") and log the
  answers as structured tags.